# GRN inference with a Laplace prior

Each gene $i$ is regressed on the regulators $j$:

$$r_i = f_i\Big(\sum_j W_{ij} h_j + b_i\Big), \qquad X_i \sim \mathrm{NB}(l \, r_i, \theta_i)$$

- $h_j$: log-CP10K of regulator $j$ (optionally standardized), $l$: library size, $\theta_i$: inverse dispersion
- $f_i = \exp$ or $f_i = \alpha_i \, \sigma$
- $W$ is fit by MAP with a Laplace prior: $\max_\theta \sum_{\text{cells}} \log p_\theta(X \mid W) + \log p(W)$

New edges are the large entries of $|W|$ that RegulonDB does not know about.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch

import grn_lasso

In [ ]:
ADATA_PATH = "/data/pboyeau/data_essential/de122_lce75/adata_de122_lce75_merged_full.h5ad"
NETWORK_TSV = "/data/pboyeau/data_essential/RegulonDB/NetworkRegulatorGene.tsv"
SYNONYMS_TSV = "/data/pboyeau/data_essential/RegulonDB/GeneProductAllIdentifiersSet.tsv"
PERT_KEY = "top_target_unthresholded"
CONTROL = "nontargeting"

device = "cuda" if torch.cuda.is_available() else "cpu"

## Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata.var_names = adata.var_names.str.lower()
adata.obs[PERT_KEY] = adata.obs[PERT_KEY].str.lower()
adata = adata[np.asarray(adata.layers["reads"].sum(1)).ravel() > 1e3]
adata

RegulonDB edges, keeping transcription factors only. The regulators of the model are the TFs measured in the data.

In [ ]:
edges = grn_lasso.load_edges(NETWORK_TSV, SYNONYMS_TSV, adata.var_names)
edges = edges[edges["regulator_type"] == "tf"]
tfs = sorted(edges["regulator"].unique())
print(f"{len(edges)} edges, {len(tfs)} TFs")

Keep the cells with a TF knocked down, and the controls.

In [ ]:
adata = adata[adata.obs[PERT_KEY].isin(tfs + [CONTROL])].copy()
adata

In [ ]:
regulator_names = tfs
# regulator_names = list(adata.var_names)  # every gene as a regulator
regulator_idx = adata.var_names.get_indexer(regulator_names)

data = grn_lasso.GRNData.from_anndata(adata, regulator_idx, PERT_KEY, standardize=True)
known_edges = grn_lasso.edge_matrix(edges, data.gene_names, regulator_names)

## Fit

In [ ]:
LAPLACE_SCALE = 1e-1  # smaller -> sparser W; worth trying a range

model = grn_lasso.GRN(regulator_idx, data.n_genes, nonlinearity="exp", init_rate=data.mean_rate())
history = grn_lasso.fit(
    model, data, laplace_scale=LAPLACE_SCALE, n_epochs=50, batch_size=1024, lr=1e-2, device=device
)
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("loss")

## Edges

In [ ]:
print("AUPRC vs RegulonDB:", grn_lasso.auprc(model, known_edges))
print("Fraction of known edges:", known_edges[model.mask.cpu().numpy() > 0].mean())

In [ ]:
grn_lasso.top_new_edges(model, known_edges, data.gene_names, regulator_names, k=30)

## Sigmoid nonlinearity

In [ ]:
model_sigmoid = grn_lasso.GRN(
    regulator_idx, data.n_genes, nonlinearity="sigmoid", init_rate=data.mean_rate()
)
history = grn_lasso.fit(
    model_sigmoid, data, laplace_scale=LAPLACE_SCALE, n_epochs=50, batch_size=1024, lr=1e-2,
    device=device,
)
print("AUPRC vs RegulonDB:", grn_lasso.auprc(model_sigmoid, known_edges))

## Ideas to try

- Use RegulonDB in the prior: a larger Laplace scale on known edges than on unknown ones.
- Then hold out part of the known edges from the prior and check whether the model recovers them.
- Replace Adam on $|W|$ with a proximal step (soft-thresholding) to get exact zeros.
- Every gene as a regulator (see the commented line above).